# ЛР-02: Лекарства в районные больницы

## Student notebook: civil 01

Этот notebook предназначен для самостоятельного решения.

Готового оптимального плана и заполненного решателя здесь нет.

## 1. Зачем нужен этот кейс

Три аптечных склада распределяют партии лекарств между четырьмя больницами.

После этого notebook-а студент должен уметь:

1. проверять, закрытая это задача или открытая;
2. правильно записывать ограничения по строкам и столбцам;
3. собирать `c`, `A_eq`, `b_eq`, `bounds` для `linprog`;
4. объяснять смысл маршрутов и итоговой стоимости.

## 2. Исходные данные

Тип задачи: **закрытая**.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Склад A | 30 |
| Склад B | 40 |
| Склад C | 35 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Больница 1 | 20 |
| Больница 2 | 25 |
| Больница 3 | 30 |
| Больница 4 | 30 |

### Матрица затрат

| Откуда / Куда | Больница 1 | Больница 2 | Больница 3 | Больница 4 |
| --- | --- | --- | --- | --- |
| Склад A | 5 | 7 | 6 | 10 |
| Склад B | 8 | 4 | 5 | 7 |
| Склад C | 6 | 6 | 4 | 5 |

## 3. Что нужно сделать

Идите тем же маршрутом, что и в теории ЛР-02:

1. Проверьте баланс запасов и спроса.
2. Если задача открытая, добавьте фиктивный узел и поясните его смысл.
3. Запишите математическую модель через переменные $x_{ij}$.
4. Сформируйте `c`, `A_eq`, `b_eq`, `bounds` для `linprog`.
5. Проверьте, что `A_eq x = b_eq` действительно задаёт строки и столбцы.
6. После собственной попытки решите задачу через Python.
7. Кратко объясните реальные и фиктивные маршруты.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9

# Шаг 1: записываем данные задачи, не меняя числовую постановку.
supplier_names = [
    'Склад A',
    'Склад B',
    'Склад C',
]
consumer_names = [
    'Больница 1',
    'Больница 2',
    'Больница 3',
    'Больница 4',
]

supplies = np.array(
    [
        30,
        40,
        35,
    ],
    dtype=float,
)

demands = np.array(
    [
        20,
        25,
        30,
        30,
    ],
    dtype=float,
)

costs = np.array(
    [
        [5, 7, 6, 10],
        [8, 4, 5, 7],
        [6, 6, 4, 5],
    ],
    dtype=float,
)

# Шаг 2: выводим векторы и матрицу затрат в читаемом виде.
supply_df = pd.DataFrame({"запас": supplies}, index=supplier_names)
demand_df = pd.DataFrame({"спрос": demands}, index=consumer_names)
cost_df = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df)

print("sum supply =", supplies.sum())
print("sum demand =", demands.sum())


Запасы поставщиков a_i:


,запас
Склад A,30.0
Склад B,40.0
Склад C,35.0


Спрос потребителей b_j:


,спрос
Больница 1,20.0
Больница 2,25.0
Больница 3,30.0
Больница 4,30.0


Матрица затрат c_ij:


,Больница 1,Больница 2,Больница 3,Больница 4
Склад A,5.0,7.0,6.0,10.0
Склад B,8.0,4.0,5.0,7.0
Склад C,6.0,6.0,4.0,5.0


sum supply = 105.0
sum demand = 105.0


## 4. Шаблон для самостоятельной сборки модели

Ниже намеренно оставлены `TODO`. Это не готовое решение, а guided skeleton:
он показывает правильные имена, порядок шагов и форму LP-модели, но ключевые
строки вы заполняете самостоятельно.


In [2]:
def balance_transport_problem(supplies, demands, costs, supplier_names, consumer_names, dummy_cost=0.0):
    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    balance_note = "Задача закрытая: сумма запасов = сумме спроса = 105"

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )

# Функция построения LP-модели
def build_transport_lp(supplies, demands, costs):
    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    # Вектор цели (стоимости перевозок)
    c = costs.flatten()

    # Матрица ограничений A_eq
    # Строки для поставщиков (supplier_count строк)
    # Строки для потребителей (consumer_count строк)
    A_eq = np.zeros((supplier_count + consumer_count, variable_count))

    # Заполняем строки поставщиков
    for i in range(supplier_count):
        for j in range(consumer_count):
            idx = i * consumer_count + j
            A_eq[i, idx] = 1

    # Заполняем строки потребителей
    for j in range(consumer_count):
        for i in range(supplier_count):
            idx = i * consumer_count + j
            A_eq[supplier_count + j, idx] = 1

    # Вектор правых частей
    b_eq = np.concatenate([supplies, demands])

    # Границы переменных
    bounds = [(0.0, None) for _ in range(variable_count)]

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "variable_count": variable_count,
    }

# Сбалансируем задачу
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print(f"Сбалансированные запасы: {balanced_supplies.sum()}")
print(f"Сбалансированный спрос: {balanced_demands.sum()}")

# Построим LP-модель
lp_model = build_transport_lp(balanced_supplies, balanced_demands, balanced_costs)

# Вывод переменных и их стоимости
route_labels = [
    f"x_{supplier_idx + 1},{consumer_idx + 1}"
    for supplier_idx in range(len(balanced_supplier_names))
    for consumer_idx in range(len(balanced_consumer_names))
]

print("\nПеременные и стоимости:")
display(pd.DataFrame({"переменная": route_labels, "стоимость c": lp_model["c"]}))

print("\nМатрица ограничений A_eq:")
display(pd.DataFrame(lp_model["A_eq"], columns=route_labels))

print("\nПравые части ограничений b_eq:")
display(pd.DataFrame({"b_eq": lp_model["b_eq"]}))

# Решение задачи
result = linprog(
    lp_model["c"],
    A_eq=lp_model["A_eq"],
    b_eq=lp_model["b_eq"],
    bounds=lp_model["bounds"],
    method="highs",
)

print(f"\nУспех решения: {result.success}")
print(f"Сообщение: {result.message}")

if result.success:
    # Преобразуем решение в матрицу плана перевозок
    plan = result.x.reshape(len(balanced_supplier_names), len(balanced_consumer_names))
    plan_df = pd.DataFrame(
        plan,
        index=balanced_supplier_names,
        columns=balanced_consumer_names,
        dtype=int
    )

    print("\nОптимальный план перевозок:")
    display(plan_df)

    # Добавим итоговые строки и столбцы
    plan_df.loc['Итого'] = plan_df.sum(axis=0)
    plan_df['Итого'] = plan_df.sum(axis=1)

    print("\nПлан перевозок с итогами:")
    display(plan_df)

    # Проверка сумм по строкам и столбцам
    print("\nПроверка ограничений:")
    print("Строки (поставщики):")
    for i, name in enumerate(balanced_supplier_names):
        row_sum = plan[i, :].sum()
        print(f"  {name}: {row_sum} = {balanced_supplies[i]} (разница: {abs(row_sum - balanced_supplies[i]):.2e})")

    print("Столбцы (потребители):")
    for j, name in enumerate(balanced_consumer_names):
        col_sum = plan[:, j].sum()
        print(f"  {name}: {col_sum} = {balanced_demands[j]} (разница: {abs(col_sum - balanced_demands[j]):.2e})")

    # Общая стоимость
    total_cost = result.fun
    print(f"\nМинимальная общая стоимость перевозок: {total_cost:.2f}")

Задача закрытая: сумма запасов = сумме спроса = 105
Сбалансированные запасы: 105.0
Сбалансированный спрос: 105.0

Переменные и стоимости:


,переменная,стоимость c
0,"x_1,1",5.0
1,"x_1,2",7.0
2,"x_1,3",6.0
3,"x_1,4",10.0
4,"x_2,1",8.0
5,"x_2,2",4.0
6,"x_2,3",5.0
7,"x_2,4",7.0
8,"x_3,1",6.0
9,"x_3,2",6.0



Матрица ограничений A_eq:


,"x_1,1","x_1,2","x_1,3","x_1,4","x_2,1","x_2,2","x_2,3","x_2,4","x_3,1","x_3,2","x_3,3","x_3,4"
0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0



Правые части ограничений b_eq:


...


Успех решения: True
Сообщение: Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальный план перевозок:


,Больница 1,Больница 2,Больница 3,Больница 4
Склад A,20,0,10,0
Склад B,0,25,15,0
Склад C,0,0,5,30



План перевозок с итогами:


,Больница 1,Больница 2,Больница 3,Больница 4,Итого
Склад A,20,0,10,0,30
Склад B,0,25,15,0,40
Склад C,0,0,5,30,35
Итого,20,25,30,30,105



Проверка ограничений:
Строки (поставщики):
  Склад A: 30.0 = 30.0 (разница: 0.00e+00)
  Склад B: 40.0 = 40.0 (разница: 0.00e+00)
  Склад C: 35.0 = 35.0 (разница: 0.00e+00)
Столбцы (потребители):
  Больница 1: 20.0 = 20.0 (разница: 0.00e+00)
  Больница 2: 25.0 = 25.0 (разница: 0.00e+00)
  Больница 3: 30.0 = 30.0 (разница: 0.00e+00)
  Больница 4: 30.0 = 30.0 (разница: 0.00e+00)

Минимальная общая стоимость перевозок: 505.00


## 5. Что должно быть в отчёте

1. Таблица исходных данных и проверка баланса.
2. Полная математическая постановка через $x_{ij}$.
3. Пояснение, нужен ли фиктивный поставщик или фиктивный потребитель.
4. Векторно-матричная запись `c`, `A_eq`, `b_eq`, `bounds`.
5. Проверка через `linprog`.
6. Таблица оптимального плана перевозок.
7. Проверка сумм по строкам и столбцам.
8. Содержательная интерпретация ненулевых маршрутов.

## 6. Контрольный чек-лист

- [ ] Я сам проверил баланс до запуска solver-а.
- [ ] Я осмысленно собрал `A_eq` и `b_eq` через индекс `route_index`.
- [ ] Я проверил `bounds` и неотрицательность всех $x_{ij}$.
- [ ] Я отличаю реальные маршруты от фиктивного узла.
- [ ] Я объяснил полученный план простыми словами.
